# LoRA GPT-2 Medium E2E Training + Evaluation on Google Colab

Use this notebook to rerun the GPT-2 Medium + LoRA + E2E NLG reproduction on a Colab GPU.

This combined notebook:

- clones or updates the project,
- downloads and preprocesses the official Microsoft LoRA E2E files,
- runs tests and dry-run checks,
- trains LoRA adapters with validation loss/perplexity logged at the end of each epoch,
- generates full E2E test predictions,
- computes grouped multi-reference E2E metrics,
- creates report figures,
- backs up outputs to Google Drive.

Recommended runtime: `Runtime > Change runtime type > GPU`. A T4 should work; L4/A100 will be faster.

## 1. Check GPU

Run this first to confirm Colab assigned a CUDA GPU.

In [ ]:
!nvidia-smi

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Clone Or Update The Repo

If the repo is private, paste a GitHub token. If it is public for your session, press Enter.

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess

REPO_OWNER = "justinlxiang"
REPO_NAME = "CS4782-final-project"
BRANCH = "main"
PROJECT_DIR = Path("/content") / REPO_NAME
WORK_DIR = PROJECT_DIR / "lora-gpt2-medium-e2e"

token = getpass("GitHub token, or press Enter for public clone: ")
repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
if token:
    repo_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print("working directory:", Path.cwd())
!git log --oneline -3

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Google Drive

Backups are written under one run folder in Drive:

```text
/content/drive/MyDrive/e2e_lora_r4_alpha32/
```

The run folder mirrors `outputs/runs/e2e_lora_r4_alpha32/` and includes checkpoints, metrics, generations, grouped E2E files, config, notebook, and `figures/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Drive once, then keep everything for this experiment inside one run folder.
drive.mount('/content/drive')

DRIVE_RUN_DIR = Path('/content/drive/MyDrive/e2e_lora_r4_alpha32')
LOCAL_RUN_DIR = Path('outputs/runs/e2e_lora_r4_alpha32')
LOCAL_ADAPTER = LOCAL_RUN_DIR / 'checkpoints' / 'adapter_final.pt'
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)


def backup_to_run(relative_path: str | Path, destination_name: str | None = None) -> Path | None:
    """Copy a file/folder into the Drive run folder."""
    source = Path(relative_path)
    if not source.exists():
        print('skip missing:', source)
        return None
    destination = DRIVE_RUN_DIR / (destination_name or source.name)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    print('backed up:', source, '->', destination)
    return destination

print('Drive run dir:', DRIVE_RUN_DIR)
print('Local run dir:', LOCAL_RUN_DIR)

## 5. Download E2E Dataset

These are the official E2E files used by the Microsoft LoRA NLG example. Each raw row has `context||completion`.

In [ ]:
!mkdir -p data/raw/e2e
!curl -L -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt
!wc -l data/raw/e2e/*.txt

## 6. Preprocess E2E

This creates `data/processed/e2e_gpt2/*.jsonl` with the official-style token sequence:

`raw_context + 50256 + leading_space_completion + 50256`

In [ ]:
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_lora.yaml

import json
from pathlib import Path

example = json.loads(Path('data/processed/e2e_gpt2/train.jsonl').read_text().splitlines()[0])
print('prompt:', example['prompt'])
print('first input ids:', example['input_ids'][:20])
print('first labels:', example['labels'][:20])
print('prompt length:', example['prompt_length'])

## 7. Run Tests And Dry Run

This verifies LoRA injection, preprocessing, checkpoint helpers, generation helpers, evaluation grouping, and one forward loss pass.

In [ ]:
!python -m pytest
!python scripts/count_params.py --config configs/e2e_gpt2_medium_lora.yaml
!python scripts/train.py --config configs/e2e_gpt2_medium_lora.yaml --dry-run --device cuda --dry-run-forward-pass

## 8. Optional: Short Smoke Training

This runs a few optimizer steps only to verify backward pass. Skip it if you are confident and want to go straight to full training.

In [ ]:
RUN_SMOKE_TRAIN = False

if RUN_SMOKE_TRAIN:
    !python scripts/train.py       --config configs/e2e_gpt2_medium_lora.yaml       --smoke-train       --device cuda       --dry-run-max-examples 80       --dry-run-batch-size 8       --smoke-max-steps 10

## 9. Full Training With End-Of-Epoch Validation

This uses the closer paper-aligned config: seed `110`, LoRA rank `4`, alpha `32`, dropout `0.1`, AdamW LR `2e-4`, epsilon `1e-6`, no gradient clipping, 5 epochs.

The training script now evaluates validation loss and perplexity after each completed epoch and appends records to `outputs/runs/e2e_lora_r4_alpha32/metrics.jsonl`.

In [ ]:
!python scripts/train.py   --config configs/e2e_gpt2_medium_lora.yaml   --train   --device cuda

## 10. Inspect Training And Validation Logs

In [ ]:
import json
from pathlib import Path

metrics_path = Path('outputs/runs/e2e_lora_r4_alpha32/metrics.jsonl')
records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
validation = [record for record in records if record.get('type') == 'validation']
print('total metric records:', len(records))
print('validation records:')
for record in validation:
    print(record)

!ls -lh outputs/runs/e2e_lora_r4_alpha32/checkpoints | tail

## 11. Back Up Training Outputs To Drive

This backs up the whole local run directory into:

```text
/content/drive/MyDrive/e2e_lora_r4_alpha32/
```

In [ ]:
backup_to_run(LOCAL_RUN_DIR, destination_name='.')
backup_to_run('configs/e2e_gpt2_medium_lora.yaml', destination_name='config_source.yaml')
backup_to_run('colab_train_lora.ipynb')

print('Training backup run folder:', DRIVE_RUN_DIR)
!du -sh /content/drive/MyDrive/e2e_lora_r4_alpha32
!find /content/drive/MyDrive/e2e_lora_r4_alpha32 -maxdepth 2 -type f | sort | tail -20

## 12. Resume Training From A Checkpoint, If Needed

Only run this if Colab disconnects before training finishes. Set `RESUME_CHECKPOINT` to the latest checkpoint in Drive or local outputs.

In [ ]:
RUN_RESUME = False
RESUME_CHECKPOINT = 'outputs/runs/e2e_lora_r4_alpha32/checkpoints/adapter_step_1000.pt'

if RUN_RESUME:
    !python scripts/train.py       --config configs/e2e_gpt2_medium_lora.yaml       --train       --device cuda       --resume-checkpoint "$RESUME_CHECKPOINT

## 13. Generate Full Test Predictions

This uses the configured `generation.decoder`, currently `official_beam`, which is closer to Microsoft `gpt2_beam.py` than Hugging Face `generate()`.

`BATCH_SIZE=4` is conservative. If you have a large GPU, try `8`; if you hit OOM, lower it.

In [ ]:
BATCH_SIZE = 4

!TOKENIZERS_PARALLELISM=false TRANSFORMERS_VERBOSITY=error python scripts/generate.py   --config configs/e2e_gpt2_medium_lora.yaml   --split test   --adapter "$LOCAL_ADAPTER"   --batch-size "$BATCH_SIZE"

!ls -lh outputs/runs/e2e_lora_r4_alpha32/generations_test.txt
!python - <<'PY'
from pathlib import Path
path = Path('outputs/runs/e2e_lora_r4_alpha32/generations_test.txt')
print('prediction lines:', sum(1 for _ in path.open()))
PY

## 14. Run Grouped E2E Metrics

The main `bleu` and `rouge_l` values are grouped by unique meaning representation with multiple references, matching the official E2E evaluation structure more closely. The `line_*` metrics are stricter debugging metrics.

In [ ]:
!python scripts/evaluate.py --config configs/e2e_gpt2_medium_lora.yaml
!cat outputs/runs/e2e_lora_r4_alpha32/generations_test.metrics.json

## 15. Create Figures

Figures are written directly inside the run folder at `outputs/runs/e2e_lora_r4_alpha32/figures`, so downloading one run folder includes its plots.

In [ ]:
!python scripts/make_figures.py   --config configs/e2e_gpt2_medium_lora.yaml   --figures-dir outputs/runs/e2e_lora_r4_alpha32/figures

!ls -lh outputs/runs/e2e_lora_r4_alpha32/figures
!cat outputs/runs/e2e_lora_r4_alpha32/figures/summary.json

## 16. Back Up Evaluation Outputs And Figures To Drive

This copies the final local run folder again, including the `figures/` subfolder created inside the run.

In [ ]:
backup_to_run(LOCAL_RUN_DIR, destination_name='.')
backup_to_run('configs/e2e_gpt2_medium_lora.yaml', destination_name='config_source.yaml')
backup_to_run('colab_train_lora.ipynb')

print('Backed up complete run to:', DRIVE_RUN_DIR)
!find /content/drive/MyDrive/e2e_lora_r4_alpha32 -maxdepth 3 -type f | sort | tail -60